# LLM Reward Shaping Tutorial

Learn how to use Large Language Models to shape rewards and improve training efficiency.

## Prerequisites

Install Ollama and pull a model:
```bash
curl https://ollama.ai/install.sh | sh
ollama pull llama3
```

In [ ]:
import sys
sys.path.insert(0, '../..')

from rl_llm_toolkit import RLEnvironment, PPOAgent, OllamaBackend, LLMRewardShaper
from rl_llm_toolkit.rewards.prompts import PromptTemplate
import numpy as np
import matplotlib.pyplot as plt

print("✅ Imports successful!")

## 1. Set Up LLM Backend

In [ ]:
try:
    llm = OllamaBackend(
        model="llama3",
        base_url="http://localhost:11434",
        temperature=0.3,
        max_tokens=10,
    )
    
    # Test connection
    test_response = llm.generate("Say 0.5", system_prompt="Respond with only a number.")
    print(f"✅ LLM backend connected!")
    print(f"Test response: {test_response.content}")
    print(f"Latency: {test_response.latency_ms:.0f}ms")
    
except Exception as e:
    print(f"❌ LLM backend error: {e}")
    print("Make sure Ollama is running: ollama serve")
    raise

## 2. Create Reward Shaper

In [ ]:
reward_shaper = LLMRewardShaper(
    llm_backend=llm,
    prompt_template="cartpole",
    llm_weight=0.3,
    env_weight=0.7,
    use_cache=True,
    cache_size=10000,
)

print("✅ Reward shaper created")
print(f"LLM weight: {reward_shaper.llm_weight}")
print(f"Env weight: {reward_shaper.env_weight}")

## 3. Test Reward Shaping

In [ ]:
# Simulate a CartPole state
state = np.array([0.1, 0.2, 0.05, 0.1])  # cart_pos, cart_vel, pole_angle, pole_vel
action = 1  # Push right
next_state = np.array([0.12, 0.25, 0.04, 0.08])
env_reward = 1.0

shaped_reward, metadata = reward_shaper.shape_reward(
    env_reward=env_reward,
    state=state,
    action=action,
    next_state=next_state,
    context={"steps": 10}
)

print("Reward Shaping Results:")
print(f"  Environment reward: {metadata['env_reward']:.3f}")
print(f"  LLM reward: {metadata['llm_reward']:.3f}")
print(f"  Shaped reward: {metadata['shaped_reward']:.3f}")
print(f"  Cached: {metadata['cached']}")

## 4. Compare Training: With vs Without LLM

In [ ]:
def train_agent(use_llm=False, timesteps=30000):
    env = RLEnvironment("CartPole-v1")
    
    agent = PPOAgent(
        env=env,
        reward_shaper=reward_shaper if use_llm else None,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        seed=42,
    )
    
    print(f"\n{'='*50}")
    print(f"Training {'WITH' if use_llm else 'WITHOUT'} LLM Reward Shaping")
    print(f"{'='*50}")
    
    results = agent.train(
        total_timesteps=timesteps,
        log_interval=5,
        progress_bar=True,
    )
    
    stats = agent.get_training_stats()
    eval_results = agent.evaluate(episodes=20, deterministic=True)
    
    env.close()
    
    return {
        'rewards': stats['stats']['episode_rewards'],
        'lengths': stats['stats']['episode_lengths'],
        'eval_mean': eval_results['mean_reward'],
        'eval_std': eval_results['std_reward'],
    }

# Train both versions
results_no_llm = train_agent(use_llm=False, timesteps=30000)
results_with_llm = train_agent(use_llm=True, timesteps=30000)

## 5. Visualize Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Smooth rewards
window = 10

def smooth(data, window_size):
    if len(data) < window_size:
        return data
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

# Plot rewards comparison
smoothed_no_llm = smooth(results_no_llm['rewards'], window)
smoothed_with_llm = smooth(results_with_llm['rewards'], window)

ax1.plot(range(len(smoothed_no_llm)), smoothed_no_llm, label='Without LLM', linewidth=2)
ax1.plot(range(len(smoothed_with_llm)), smoothed_with_llm, label='With LLM', linewidth=2)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward (smoothed)')
ax1.set_title('Training Progress Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot evaluation results
methods = ['Without LLM', 'With LLM']
means = [results_no_llm['eval_mean'], results_with_llm['eval_mean']]
stds = [results_no_llm['eval_std'], results_with_llm['eval_std']]

ax2.bar(methods, means, yerr=stds, capsize=10, alpha=0.7)
ax2.set_ylabel('Mean Reward')
ax2.set_title('Final Evaluation Performance')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nFinal Results:")
print(f"Without LLM: {results_no_llm['eval_mean']:.2f} ± {results_no_llm['eval_std']:.2f}")
print(f"With LLM: {results_with_llm['eval_mean']:.2f} ± {results_with_llm['eval_std']:.2f}")

improvement = results_with_llm['eval_mean'] - results_no_llm['eval_mean']
print(f"\n📈 Improvement: {improvement:+.2f}")

## 6. LLM Usage Statistics

In [ ]:
stats = reward_shaper.get_statistics()

print("LLM Reward Shaping Statistics:")
print(f"  Cache size: {stats['cache_size']}")
print(f"  Cache hits: {stats['cache_hits']}")
print(f"  Cache misses: {stats['cache_misses']}")
print(f"  Cache hit rate: {stats['cache_hit_rate']:.2%}")
print(f"\nLLM API Statistics:")
print(f"  Total requests: {stats['llm_stats']['total_requests']}")
print(f"  Total tokens: {stats['llm_stats']['total_tokens']}")
print(f"  Avg tokens/request: {stats['llm_stats']['avg_tokens_per_request']:.1f}")

# Calculate cost savings from caching
total_calls = stats['cache_hits'] + stats['cache_misses']
if total_calls > 0:
    savings = (stats['cache_hits'] / total_calls) * 100
    print(f"\n💰 Cost savings from caching: {savings:.1f}%")

## 7. Custom Prompt Templates

In [ ]:
# Create a custom prompt template
custom_template = PromptTemplate(
    system_prompt=(
        "You are an expert RL reward designer. "
        "Evaluate actions based on stability and efficiency. "
        "Respond with ONLY a number between -1 and 1."
    ),
    user_template=(
        "CartPole State:\n"
        "  Cart Position: {cart_pos:.3f}\n"
        "  Pole Angle: {pole_angle:.3f} rad\n"
        "  Action: {'Left' if action == 0 else 'Right'}\n"
        "  Steps: {steps}\n\n"
        "Rate this action (-1 to 1):"
    )
)

# Create shaper with custom template
custom_shaper = LLMRewardShaper(
    llm_backend=llm,
    prompt_template=custom_template,
    llm_weight=0.4,
    env_weight=0.6,
)

# Test it
state = np.array([0.05, 0.1, 0.02, 0.05])
shaped_reward, metadata = custom_shaper.shape_reward(
    env_reward=1.0,
    state=state,
    action=1,
    next_state=state,
    context={
        'cart_pos': state[0],
        'pole_angle': state[2],
        'action': 1,
        'steps': 50
    }
)

print("Custom Template Results:")
print(f"  LLM reward: {metadata['llm_reward']:.3f}")
print(f"  Shaped reward: {metadata['shaped_reward']:.3f}")

## 8. Experiment with Different LLM Weights

In [ ]:
def test_llm_weight(llm_weight, timesteps=20000):
    env = RLEnvironment("CartPole-v1")
    
    shaper = LLMRewardShaper(
        llm_backend=llm,
        prompt_template="cartpole",
        llm_weight=llm_weight,
        env_weight=1.0 - llm_weight,
        use_cache=True,
    )
    
    agent = PPOAgent(env=env, reward_shaper=shaper, seed=42)
    agent.train(total_timesteps=timesteps, log_interval=100, progress_bar=False)
    
    eval_results = agent.evaluate(episodes=10, deterministic=True)
    env.close()
    
    return eval_results['mean_reward']

# Test different weights
weights = [0.0, 0.2, 0.3, 0.4, 0.5]
results = []

print("Testing different LLM weights...\n")
for w in weights:
    print(f"Testing LLM weight = {w}...")
    mean_reward = test_llm_weight(w)
    results.append(mean_reward)
    print(f"  Mean reward: {mean_reward:.2f}\n")

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(weights, results, marker='o', linewidth=2, markersize=8)
plt.xlabel('LLM Weight')
plt.ylabel('Mean Reward')
plt.title('Performance vs LLM Weight')
plt.grid(True, alpha=0.3)
plt.show()

best_idx = np.argmax(results)
print(f"\n🏆 Best LLM weight: {weights[best_idx]} (reward: {results[best_idx]:.2f})")

## 9. Key Takeaways

1. **LLM reward shaping** can improve training efficiency
2. **Caching** reduces API costs by 80%+
3. **Weight tuning** is important (typically 0.2-0.4 for LLM weight)
4. **Custom prompts** allow domain-specific knowledge injection
5. **Local LLMs** (Ollama) provide free experimentation

## Next Steps

- Try LLM shaping on harder environments
- Experiment with different LLM models
- Create domain-specific prompt templates
- Compare OpenAI vs Ollama backends